# 03 · Train and evaluate ACT (GPU runtime, Colab or Kaggle)

One run per dataset. Set `DATASET` and run all cells. On Kaggle, use "Save Version → Save & Run All", which keeps running after you close the tab.

| run | DATASET | WEIGHTS |
|---|---|---|
| ACT on official demos | `official` | – |
| ACT on ours, matched count | `ours_50` | – |
| ACT on ours, all | `ours_all` | – |
| advantage-weighted ACT (offline RL, gated) | `ours_mix` | `weights.npz` from notebook 04 |

Evaluation: the same 50 fixed LIBERO scenes for every run, with a 95% Wilson interval.

In [ ]:
import os, sys, json, subprocess
os.environ["MUJOCO_GL"] = "egl"                      # GPU rendering, must be set before mujoco is imported
REPO_URL = "https://github.com/LukasWill/phone2panda.git"
BASE = "/content" if os.path.exists("/content") else "/kaggle/working"   # Colab or Kaggle
CODE_DIR = f"{BASE}/phone2panda"
if not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, CODE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CODE_DIR, "pull", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lerobot[libero]==0.6.1", "imageio-ffmpeg"], check=True)
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
# Hugging Face token (Colab: key icon -> Secrets -> HF_TOKEN; Kaggle: Add-ons -> Secrets -> HF_TOKEN)
try:
    from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient; os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN found: uploads to the Hub will fail")
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
DATASET = "ours_all"
WEIGHTS = None            # e.g. "rl/weights.npz" for advantage-weighted ACT
STEPS = 20000
BATCH = 32
SEED = 0
HUB_REPO = "LukasWill/phone2panda-data"
RUN = f"act_{DATASET}" + ("_awr" if WEIGHTS else "") + f"_s{SEED}"

In [ ]:
from p2p.hub import pull
pull(HUB_REPO, f"{BASE}/hub", allow_patterns=[f"lerobot/{DATASET}/*"] + ([WEIGHTS] if WEIGHTS else []))
DATA = f"{BASE}/hub/lerobot/{DATASET}"
W = f"--weights {BASE}/hub/{WEIGHTS}" if WEIGHTS else ""
!python -m p2p.policy.act train {DATA} {BASE}/runs/{RUN} --steps {STEPS} --batch-size {BATCH} --seed {SEED} {W}

In [ ]:
!python -m p2p.policy.act eval {BASE}/runs/{RUN}/final --episodes 50 --videos 3 --out {BASE}/runs/{RUN}/eval
print(json.load(open(f"{BASE}/runs/{RUN}/eval/eval.json"))["rate"])

In [ ]:
from p2p.hub import push
push(f"{BASE}/runs/{RUN}", HUB_REPO, path_in_repo=f"runs/{RUN}")